# Lecture 18: Productionizing NLP Systems
### NLP Course 2027

---

## Learning Outcomes
- Design and deploy a REST API for NLP model serving
- Optimize models for production: quantization, caching
- Monitor NLP systems in production: drift, latency, errors
- Apply MLOps practices: experiment tracking, versioning

**Primary Reference:** *Practical NLP* Ch.11

## 1. The Gap Between Research and Production

```
 Research prototype          Production system
 ─────────────────          ─────────────────
 Jupyter notebook       ->   REST API / microservice
 Single model           ->   Load-balanced replicas
 No versioning          ->   Model registry + CI/CD
 Manual evaluation      ->   Continuous monitoring
 GPU server             ->   Optimized inference
 Static dataset         ->   Live data + drift detection
 No error handling      ->   Graceful degradation
```

*Practical NLP* emphasizes: 'A model that isn't deployed is not useful to anyone. The goal is to get NLP into real products.'

## 2. Serving NLP Models with FastAPI

FastAPI is the recommended framework for production NLP APIs:
- Fast (async, based on Starlette)
- Automatic OpenAPI/Swagger documentation
- Type validation with Pydantic
- Easy to containerize

In [4]:
# FastAPI NLP service
# !pip install fastapi uvicorn

fastapi_app = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline
from typing import List
import time
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(
    title="NLP API",
    description="Production NLP service for text classification and NER",
    version="1.0.0"
)

# Load models at startup (not per request!)
@app.on_event("startup")
async def load_models():
    app.state.sentiment = pipeline(
        "text-classification",
        model="distilbert-base-uncased-finetuned-sst-2-english"
    )
    app.state.ner = pipeline(
        "ner",
        model="dbmdz/bert-large-cased-finetuned-conll03-english",
        aggregation_strategy="simple"
    )
    logger.info("Models loaded successfully")

class TextRequest(BaseModel):
    text: str
    class Config:
        schema_extra = {"example": {"text": "I love this NLP course!"}}

class BatchRequest(BaseModel):
    texts: List[str]

class SentimentResponse(BaseModel):
    label: str
    score: float
    latency_ms: float

@app.post("/sentiment", response_model=SentimentResponse)
async def predict_sentiment(req: TextRequest):
    if not req.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    start = time.time()
    result = app.state.sentiment(req.text)[0]
    latency = (time.time() - start) * 1000
    logger.info(f"Sentiment: {result[\"label\"]} for text: {req.text[:50]}")
    return SentimentResponse(
        label=result["label"],
        score=result["score"],
        latency_ms=round(latency, 2)
    )

@app.post("/ner")
async def predict_ner(req: TextRequest):
    entities = app.state.ner(req.text)
    return {"entities": entities, "count": len(entities)}

@app.get("/health")
async def health_check():
    return {"status": "healthy", "models_loaded": True}

# To run: uvicorn app:app --host 0.0.0.0 --port 8000
'''
print('FastAPI service code:')
print(fastapi_app)

FastAPI service code:

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline
from typing import List
import time
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(
    title="NLP API",
    description="Production NLP service for text classification and NER",
    version="1.0.0"
)

# Load models at startup (not per request!)
@app.on_event("startup")
async def load_models():
    app.state.sentiment = pipeline(
        "text-classification",
        model="distilbert-base-uncased-finetuned-sst-2-english"
    )
    app.state.ner = pipeline(
        "ner",
        model="dbmdz/bert-large-cased-finetuned-conll03-english",
        aggregation_strategy="simple"
    )
    logger.info("Models loaded successfully")

class TextRequest(BaseModel):
    text: str
    class Config:
        schema_extra = {"example": {"text": "I love this NLP course!"}}

class BatchRequest(BaseModel):
    tex

In [5]:
# Client code to call the API
import json

client_code = '''
import requests

BASE_URL = "http://localhost:8000"

# Health check
response = requests.get(f"{BASE_URL}/health")
print("Health:", response.json())

# Sentiment analysis
text = "The transformer architecture has revolutionized NLP!"
response = requests.post(
    f"{BASE_URL}/sentiment",
    json={"text": text}
)
result = response.json()
print(f"Sentiment: {result[\'label\']} ({result[\'score\']:.3f}) in {result[\'latency_ms\']:.1f}ms")

# Batch processing
texts = ["Great product!", "Terrible service.", "It was okay."]
for text in texts:
    r = requests.post(f"{BASE_URL}/sentiment", json={"text": text})
    res = r.json()
    print(f"{res[\'label\']:10s} {text}")
'''
print('Client code:')
print(client_code)

Client code:

import requests

BASE_URL = "http://localhost:8000"

# Health check
response = requests.get(f"{BASE_URL}/health")
print("Health:", response.json())

# Sentiment analysis
text = "The transformer architecture has revolutionized NLP!"
response = requests.post(
    f"{BASE_URL}/sentiment",
    json={"text": text}
)
result = response.json()
print(f"Sentiment: {result['label']} ({result['score']:.3f}) in {result['latency_ms']:.1f}ms")

# Batch processing
texts = ["Great product!", "Terrible service.", "It was okay."]
for text in texts:
    r = requests.post(f"{BASE_URL}/sentiment", json={"text": text})
    res = r.json()
    print(f"{res['label']:10s} {text}")



## 3. Model Optimization for Production

### Optimization Techniques

| Technique | Speed Gain | Size Reduction | Accuracy Loss |
|-----------|-----------|----------------|---------------|
| Dynamic quantization | 2-4x | 4x | <1% |
| ONNX export | 2-5x | Similar | None |
| TorchScript | 1.5-3x | Similar | None |
| Model pruning | 2-3x | 2-10x | 1-3% |
| DistilBERT | 2x | 40% | 3-5% |
| Caching | N/A (repeat queries) | - | None |

In [6]:
# Model serialization and optimization
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = 'distilbert-base-uncased-finetuned-sst-2-english'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.eval()

# Method 1: Save with PyTorch
torch.save(model.state_dict(), '/tmp/model_weights.pt')
print('Saved PyTorch weights to /tmp/model_weights.pt')

# Method 2: Save full model
model.save_pretrained('/tmp/sentiment_model')
tokenizer.save_pretrained('/tmp/sentiment_model')
print('Saved HF model to /tmp/sentiment_model/')

# Method 3: TorchScript (for C++ deployment)
# strict=False required because transformer models return dict-like outputs
text = 'This is a test'
encoded = tokenizer(text, return_tensors='pt')
with torch.no_grad():
    traced = torch.jit.trace(model, [encoded['input_ids'], encoded['attention_mask']], strict=False)
torch.jit.save(traced, '/tmp/model_traced.pt')
print('Saved TorchScript model to /tmp/model_traced.pt')

Saved PyTorch weights to /tmp/model_weights.pt
Saved HF model to /tmp/sentiment_model/
Saved TorchScript model to /tmp/model_traced.pt


In [7]:
# ONNX export for cross-platform inference
# !pip install onnx onnxruntime optimum

onnx_export_code = '''
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

# Export to ONNX
model = ORTModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english",
    export=True
)
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english")

# Save
model.save_pretrained("./onnx_model")
tokenizer.save_pretrained("./onnx_model")

# Fast inference with ONNX Runtime
from transformers import pipeline
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer)
result = pipe("This is great!")
print(result)  # Same API, ~2-5x faster on CPU
'''
print('ONNX export code (requires optimum):')
print(onnx_export_code)

ONNX export code (requires optimum):

from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

# Export to ONNX
model = ORTModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english",
    export=True
)
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased-finetuned-sst-2-english")

# Save
model.save_pretrained("./onnx_model")
tokenizer.save_pretrained("./onnx_model")

# Fast inference with ONNX Runtime
from transformers import pipeline
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer)
result = pipe("This is great!")
print(result)  # Same API, ~2-5x faster on CPU



In [8]:
# Prediction caching with LRU cache
from functools import lru_cache
from transformers import pipeline
import time

sentiment = pipeline('text-classification',
                     model='distilbert-base-uncased-finetuned-sst-2-english')

@lru_cache(maxsize=1000)
def cached_predict(text: str):
    """Cache predictions for repeated queries."""
    return sentiment(text)[0]

# Test caching
texts = [
    'I love this product!',
    'Terrible service.',
    'I love this product!',  # repeat - should be instant
    'I love this product!',  # repeat
]
for text in texts:
    t0 = time.time()
    result = cached_predict(text)
    latency = (time.time() - t0) * 1000
    cache_hit = '(cache hit)' if latency < 5 else '(model inference)'
    print(f'{result["label"]:10s} {latency:6.1f}ms {cache_hit} | {text}')

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


POSITIVE     44.8ms (model inference) | I love this product!
NEGATIVE     18.1ms (model inference) | Terrible service.
POSITIVE      0.0ms (cache hit) | I love this product!
POSITIVE      0.0ms (cache hit) | I love this product!


## 4. Monitoring in Production

### What to Monitor
```
1. Service health:    latency, throughput, error rates
2. Model performance: accuracy on sampled + labeled traffic
3. Data drift:        input distribution shifting over time
4. Prediction drift:  output distribution changing
```

### Data Drift
When the input distribution shifts, model performance degrades silently.
Example: Sentiment model trained on 2020 reviews deployed in 2025 - new slang, new topics.

### Monitoring Tools
- **Prometheus + Grafana**: infrastructure metrics
- **Evidently**: ML-specific drift detection
- **MLflow**: experiment tracking and model registry
- **Weights & Biases (W&B)**: experiment logging

In [9]:
# MLflow experiment tracking
# !pip install mlflow

mlflow_code = '''
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

mlflow.set_experiment("nlp-sentiment-classifier")

# Log parameters, metrics, and model
with mlflow.start_run(run_name="tfidf-logistic-v1"):
    # Parameters
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_features", 10000)
    mlflow.log_param("max_iter", 500)
    mlflow.log_param("ngram_range", "1,2")

    # Train model
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1,2))),
        ("clf", LogisticRegression(max_iter=500))
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_macro", f1)

    # Save model
    mlflow.sklearn.log_model(pipeline, "model")
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")
'''
print('MLflow tracking code:')
print(mlflow_code)

MLflow tracking code:

import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

mlflow.set_experiment("nlp-sentiment-classifier")

# Log parameters, metrics, and model
with mlflow.start_run(run_name="tfidf-logistic-v1"):
    # Parameters
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("max_features", 10000)
    mlflow.log_param("max_iter", 500)
    mlflow.log_param("ngram_range", "1,2")

    # Train model
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1,2))),
        ("clf", LogisticRegression(max_iter=500))
    ])
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")
    mlflow.log_metric("accuracy", acc)
    mlflo

In [10]:
# Data drift detection concept
import numpy as np
from scipy import stats

def detect_text_drift(reference_texts, current_texts, feature='length'):
    """
    Detect distribution shift between reference and current batches.
    Uses Kolmogorov-Smirnov test on text length distribution.
    """
    if feature == 'length':
        ref_values = np.array([len(t.split()) for t in reference_texts])
        cur_values = np.array([len(t.split()) for t in current_texts])
    elif feature == 'char_length':
        ref_values = np.array([len(t) for t in reference_texts])
        cur_values = np.array([len(t) for t in current_texts])

    stat, p_value = stats.ks_2samp(ref_values, cur_values)
    drift_detected = p_value < 0.05

    return {
        'feature': feature,
        'ks_statistic': round(stat, 4),
        'p_value': round(p_value, 4),
        'drift_detected': drift_detected,
        'ref_mean': round(ref_values.mean(), 2),
        'cur_mean': round(cur_values.mean(), 2),
    }

# Simulate: training data was short reviews, now getting long ones
np.random.seed(42)
reference = ['short review ' * np.random.randint(5, 15) for _ in range(200)]
current_no_drift = ['short review ' * np.random.randint(5, 15) for _ in range(100)]
current_with_drift = ['long review with lots of details ' * np.random.randint(20, 50) for _ in range(100)]

print('Drift Detection Results:')
print('No drift scenario:')
print(detect_text_drift(reference, current_no_drift))
print('\nWith drift (longer texts):')
print(detect_text_drift(reference, current_with_drift))

Drift Detection Results:
No drift scenario:
{'feature': 'length', 'ks_statistic': 0.135, 'p_value': 0.171, 'drift_detected': False, 'ref_mean': 19.12, 'cur_mean': 17.72}

With drift (longer texts):
{'feature': 'length', 'ks_statistic': 1.0, 'p_value': 0.0, 'drift_detected': True, 'ref_mean': 19.12, 'cur_mean': 209.64}


In [11]:
# Complete monitoring wrapper
import time
import json
from datetime import datetime
from collections import deque

class ModelMonitor:
    """Production model monitor tracking predictions and performance."""

    def __init__(self, model_name, window_size=1000):
        self.model_name = model_name
        self.predictions = deque(maxlen=window_size)
        self.latencies = deque(maxlen=window_size)
        self.errors = deque(maxlen=window_size)

    def record_prediction(self, input_text, prediction, confidence, latency_ms):
        self.predictions.append({
            'timestamp': datetime.utcnow().isoformat(),
            'input_length': len(input_text.split()),
            'prediction': prediction,
            'confidence': confidence,
            'latency_ms': latency_ms,
        })
        self.latencies.append(latency_ms)

    def record_error(self, error_type, error_msg):
        self.errors.append({
            'timestamp': datetime.utcnow().isoformat(),
            'error_type': error_type,
            'message': str(error_msg)[:200],
        })

    def get_stats(self):
        if not self.latencies:
            return {}
        preds = list(self.predictions)
        label_dist = {}
        for p in preds:
            label_dist[p['prediction']] = label_dist.get(p['prediction'], 0) + 1
        return {
            'model': self.model_name,
            'total_predictions': len(preds),
            'avg_latency_ms': round(sum(self.latencies) / len(self.latencies), 2),
            'p95_latency_ms': round(sorted(self.latencies)[int(len(self.latencies)*0.95)], 2),
            'error_count': len(self.errors),
            'label_distribution': label_dist,
        }

# Demo
monitor = ModelMonitor('sentiment-v1.2')

# Simulate predictions
from transformers import pipeline
sentiment = pipeline('text-classification', model='distilbert-base-uncased-finetuned-sst-2-english')

texts = [
    'This is amazing!', 'Terrible product.', 'It was okay.',
    'Absolutely love it!', 'Would not recommend.', 'Pretty good value.'
]
for text in texts:
    t0 = time.time()
    result = sentiment(text)[0]
    latency = (time.time() - t0) * 1000
    monitor.record_prediction(text, result['label'], result['score'], latency)

print('Monitoring Stats:')
print(json.dumps(monitor.get_stats(), indent=2))

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Monitoring Stats:
{
  "model": "sentiment-v1.2",
  "total_predictions": 6,
  "avg_latency_ms": 17.44,
  "p95_latency_ms": 28.02,
  "error_count": 0,
  "label_distribution": {
    "POSITIVE": 4,
    "NEGATIVE": 2
  }
}


## Practice Exercises

See **`Lecture-18-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Stage | Tool | Key Metric |
|-------|------|------------|
| Serving | FastAPI + uvicorn | Latency, throughput |
| Optimization | ONNX, quantization | Speedup factor |
| Caching | LRU / Redis | Cache hit rate |
| Monitoring | MLflow, Evidently | Drift score, accuracy |
| Versioning | Model registry | Model version |

**Next Lecture**: Capstone & Future Trends -- LLMs, RAG, agents, ethics.

---
*Book reference: Practical NLP Ch.11*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**